# Regional Heat-Stress Vulnerability Profiles for Priority Mapping Through Clustering
**Assignment Project No. 25** | **Mapped ID**: SIH26083 | **Type**: Individual  
**Course Outcomes**: CO2, CO3, CO5  
**Tools & Topics**: Partition Clustering (K-Means), Hierarchical Comparison, PCA Visualisation, Priority Mapping  
**Dataset Proxy**: NOAA Global Summary of the Day (GSOD) Surface Weather Observations  

---

## Executive Summary & Research Context
Rising global surface temperatures and intensifying pre-monsoon heatwaves present severe biometeorological threats across diverse geographical zones. Rather than relying on simple ambient temperature alone—which fails to capture human physiological heat strain caused by humidity, boundary layer wind stagnation, and nocturnal cooling deficit—this project implements an unsupervised machine learning methodology.

Using NOAA Global Summary of the Day (GSOD) observations as the official proxy dataset, we construct an end-to-end analytical pipeline encompassing:
1. **Data Preprocessing & Standardization (CO2)**: Cleaning NOAA sensor flags, median imputation, metric SI conversions, and feature engineering (Magnus-Tetens Relative Humidity, Rothfusz NOAA Heat Index, Diurnal Temperature Range, and Canadian Humidex).
2. **Partition Clustering & Hierarchical Comparison (CO3)**: Multi-$K$ parameter sweeps ($K \in [2, 8]$) systematically comparing Within-Cluster Sum of Squares (WCSS), Silhouette Coefficients, Davies–Bouldin Indices, and Agglomerative Hierarchical Clustering concordances (Ward linkage, Cophenetic correlation, Adjusted Rand Index).
3. **Dimensionality Reduction & Priority Mapping (CO5)**: Principal Component Analysis (PCA) 2D biplot projection to identify primary thermodynamic axes, followed by biometeorological cluster profiling (Low, Moderate, High, Extreme) and geographic priority mapping for targeted public-health interventions.


## Part 1: Data Ingestion, Preprocessing & Feature Engineering (CO2)
In this section, we ingest authentic NOAA GSOD surface weather records across diverse climatic zones (Continental Desert, Indo-Gangetic Plain, Humid Coastal Trap, Deccan Plateau, and Alpine Refugia). We address NOAA missing flags (`9999.9`, `999.9`), convert imperial units into metric standard units, and derive thermodynamic indicators of heat strain.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.append(os.path.abspath(".."))

from backend.ml.data_loader import load_noaa_gsod
from backend.ml.preprocessing import NOAADataPreprocessor
from backend.ml.feature_engineering import HeatStressFeatureEngineer
from backend.ml.kmeans import KMeansClusterEngine
from backend.ml.hierarchical import HierarchicalClusterEngine
from backend.ml.validation import OptimalKSelector
from backend.ml.pca_analysis import PCAReducer
from backend.ml.profiles import VulnerabilityProfiler
from backend.ml.explainability import StationExplainer
from backend.ml.temporal_analysis import TemporalAnalyzer
from backend.ml.ai_insights import AIInsightsEngine

# Set visualization styles
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Libraries successfully imported.")


### 1.1 Ingesting NOAA GSOD Raw Observations
We load the meteorological observations and inspect the raw schema.


In [ ]:
# Ingest NOAA GSOD dataset
raw_df = load_noaa_gsod()
print(f"Total Observations: {len(raw_df):,}")
print(f"Unique Ground Stations: {raw_df['STATION'].nunique()}")
print(f"Date Range: {raw_df['DATE'].min()} to {raw_df['DATE'].max()}")
raw_df.head(5)


**Analysis of Raw Dataset**:
The ingested NOAA GSOD dataset contains raw surface observations recorded in imperial units (Fahrenheit, knots, inches) along with NOAA missing flag conventions (`9999.9`). In the following step, we perform median-based imputation, convert temperatures to Celsius ($^\circ\text{C}$), wind speed to $\text{km/h}$, and extract temporal indicators.


In [ ]:
# Initialize preprocessor and transform to clean SI metric units
preprocessor = NOAADataPreprocessor()
cleaned_df = preprocessor.prepare_raw_data(raw_df)

print("Preprocessing complete. Sample cleaned metric observations:")
cleaned_df[['STATION', 'NAME', 'LATITUDE', 'LONGITUDE', 'mean_temp_c', 'max_c', 'min_c', 'dewp_c', 'wind_speed_kmh', 'pressure_hpa']].head(5)


### 1.1b Data Quality & Station Coverage Audit
Rigorous data verification evaluating sensor missing flags, physical bounds violations, and multi-year coverage.


In [ ]:
# Execute Data Quality Audit
quality_metrics = preprocessor.calculate_data_quality_metrics(raw_df)

print("=== NOAA GSOD DATA QUALITY AUDIT ===")
print(f"Total Ingested Observations: {quality_metrics['total_records']:,}")
print(f"Unique Ground Monitoring Stations: {quality_metrics['unique_stations']}")
print(f"Observation Years: {quality_metrics['recorded_years']}")
print(f"Missing Values Imputed: {quality_metrics['missing_values_count']} ({quality_metrics['missing_values_pct']}%)")
print(f"Physical Outliers Treated: {quality_metrics['outliers_count']} ({quality_metrics['outliers_pct']}%)")
print(f"Clean Observations Retained: {quality_metrics['clean_records_count']:,}")
print(f"Overall Data Quality Score: {quality_metrics['data_quality_score']}%")


### 1.2 Biometeorological Feature Engineering
We compute scientifically rigorous biometeorological indices:
1. **Relative Humidity ($RH$)**: Derived via the **Magnus-Tetens empirical formulation**:
   $$RH = 100 \times \frac{\exp\left(\frac{17.625 \cdot T_{dew}}{243.04 + T_{dew}}\right)}{\exp\left(\frac{17.625 \cdot T}{243.04 + T}\right)}$$
2. **NOAA Heat Index ($HI$)**: Rothfusz regression apparent temperature measuring combined thermal strain.
3. **Diurnal Temperature Range ($DTR$)**: $T_{max} - T_{min}$, delineating dry continental extremes from nocturnal humid traps.
4. **Canadian Humidex**: Perceived equivalent temperature incorporating vapor pressure.


In [ ]:
engineer = HeatStressFeatureEngineer()
engineered_df = engineer.engineer_features(cleaned_df)

feature_matrix, feature_cols = engineer.get_clustering_matrix(engineered_df)
print(f"Engineered {len(feature_cols)} clustering features:")
for col in feature_cols:
    print(f" - {col}")

engineered_df[['NAME', 'mean_temp_c', 'max_temp_c', 'temperature_range', 'relative_humidity', 'heat_index_c', 'humidex_c']].head(5)


### 1.3 Feature Distributions & Correlation Matrix
Before clustering, we examine feature correlations to understand how humidity, heat index, and ambient temperature interrelate.


In [ ]:
plt.figure(figsize=(10, 8))
corr = feature_matrix.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap='coolwarm', cbar_kws={'label': 'Pearson Correlation'})
plt.title("Correlation Matrix of Biometeorological Features", fontsize=14, pad=15)
plt.tight_layout()
plt.show()


### 1.4 Feature Standardization with `StandardScaler`
K-Means is an isotropic, distance-based algorithm using Euclidean metric:
$$d(x, y) = \sqrt{\sum_{i=1}^p (x_i - y_i)^2}$$
Because variables have radically differing scales (e.g. Atmospheric Pressure $\sim 1000\text{ hPa}$ vs Temperature $\sim 35^\circ\text{C}$), unscaled features with larger numeric variance would completely dominate the partition boundaries. We apply `StandardScaler` to ensure zero mean ($\mu = 0$) and unit variance ($\sigma^2 = 1$).


In [ ]:
X_scaled, scaler = preprocessor.fit_transform_features(feature_matrix, feature_cols)
print(f"Scaled feature matrix shape: {X_scaled.shape}")
print(f"Feature Means (approx 0): {np.round(X_scaled.mean(axis=0), 3)}")
print(f"Feature Standard Deviations (approx 1): {np.round(X_scaled.std(axis=0), 3)}")


## Part 2: Partition Clustering, Model Validation & Hierarchical Comparison (CO3)
In this section, we apply K-Means clustering across multiple configurations ($K \in [2, 8]$) and quantitatively evaluate:
- **WCSS (Within-Cluster Sum of Squares / Inertia)** for the Elbow method.
- **Silhouette Coefficient ($S$)**: Measures how similar an object is to its own cluster compared to neighboring clusters ($\in [-1, 1]$).
- **Davies–Bouldin Index ($DB$)**: Measures the ratio of within-cluster scatter to between-cluster separation (lower is superior).
- **Calinski–Harabasz Index ($CH$)**: Ratio of between-cluster dispersion to within-cluster dispersion (higher is superior).
- **Automated Multi-Criteria Decision Engine**: Algorithmically selecting optimal $K$.
- **Hierarchical Clustering Comparison**: Evaluating Ward's Agglomerative Clustering against K-Means via Cophenetic correlation and Adjusted Rand Index.


In [ ]:
kmeans_engine = KMeansClusterEngine(random_state=42)
evaluations = kmeans_engine.evaluate_multi_k(X_scaled, k_range=range(2, 9))

eval_df = pd.DataFrame(evaluations)
eval_df[['k', 'wcss', 'silhouette_score', 'davies_bouldin_index', 'calinski_harabasz_index']]


### 2.1 Multi-Metric Validation Curves (Elbow, Silhouette, Davies-Bouldin)
We visualize the trade-offs across all candidate $K$ values.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Elbow Method (WCSS)
axes[0].plot(eval_df['k'], eval_df['wcss'], marker='o', color='#2563EB', linewidth=2.5, markersize=8)
axes[0].set_title("Elbow Method: WCSS vs K", fontsize=13, fontweight='bold')
axes[0].set_xlabel("Number of Clusters (K)")
axes[0].set_ylabel("Within-Cluster Sum of Squares (Inertia)")
axes[0].grid(True, alpha=0.3)

# Plot 2: Silhouette Score
axes[1].plot(eval_df['k'], eval_df['silhouette_score'], marker='s', color='#10B981', linewidth=2.5, markersize=8)
axes[1].set_title("Silhouette Score vs K (Higher is Better)", fontsize=13, fontweight='bold')
axes[1].set_xlabel("Number of Clusters (K)")
axes[1].set_ylabel("Average Silhouette Coefficient")
axes[1].grid(True, alpha=0.3)

# Plot 3: Davies-Bouldin Index
axes[2].plot(eval_df['k'], eval_df['davies_bouldin_index'], marker='^', color='#EF4444', linewidth=2.5, markersize=8)
axes[2].set_title("Davies-Bouldin Index vs K (Lower is Better)", fontsize=13, fontweight='bold')
axes[2].set_xlabel("Number of Clusters (K)")
axes[2].set_ylabel("Davies-Bouldin Index")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### 2.2 Automated Optimal K Selection
Rather than selecting $K$ purely based on visual inspection, we utilize a multi-criteria optimization function combining normalized Silhouette score, inverted Davies-Bouldin score, and knee curvature.


In [ ]:
k_selector = OptimalKSelector()
optimal_result = k_selector.select_optimal_k(evaluations)
best_k = optimal_result['optimal_k']

print("=== ALGORITHMIC OPTIMAL K RECOMMENDATION ===")
print(f"Recommended K: {best_k}")
print(f"Rationale: {optimal_result['rationale']}")

# Train the finalized model on optimal K
labels_opt, final_kmeans = kmeans_engine.fit_predict_k(X_scaled, k=best_k)
engineered_df['cluster'] = labels_opt


### 2.2b Cluster Stability Evaluation (20 Random Seed Iterations)
To verify that our discovered cluster partitions represent robust biometeorological structures rather than random seed initialization artifacts (local minima of K-Means), we execute 20 iterations with different random seeds ($s \in [1, 20]$) and compute pairwise Adjusted Rand Index (ARI).


In [ ]:
stability_res = kmeans_engine.evaluate_cluster_stability(X_scaled, k=best_k, n_seeds=20)

print("=== CLUSTER PARTITION STABILITY (20 RANDOM SEEDS) ===")
print(f"Tested Clusters (K): {stability_res['k']}")
print(f"Iterations Evaluated: {stability_res['n_seeds_tested']}")
print(f"Mean Pairwise ARI: {stability_res['mean_pairwise_ari']:.4f}")
print(f"Min / Max Pairwise ARI: {stability_res['min_pairwise_ari']:.4f} / {stability_res['max_pairwise_ari']:.4f}")
print(f"Stability Tier: {stability_res['stability_tier']} ({stability_res['stability_percentage']}%)")
print(f"Scientific Rationale: {stability_res['scientific_rationale']}")


### 2.3 Hierarchical Comparison (Agglomerative Clustering vs K-Means)
To fulfill the specific syllabus topic **'Hierarchical Comparison'**, we run Agglomerative Hierarchical Clustering using Ward's minimum variance criterion on the identical scaled dataset. We compute the **Cophenetic Correlation Coefficient** and compare cluster partitions using the **Adjusted Rand Index (ARI)**.


In [ ]:
hier_engine = HierarchicalClusterEngine(linkage_method="ward")
hier_comparison = hier_engine.compare_with_kmeans(X_scaled, labels_opt, n_clusters=best_k)

print("=== HIERARCHICAL VS PARTITION CLUSTERING COMPARISON ===")
print(f"Linkage Method: {hier_comparison['linkage_method']}")
print(f"Cophenetic Correlation: {hier_comparison['cophenetic_correlation']}")
print(f"Adjusted Rand Index (ARI): {hier_comparison['adjusted_rand_index']}")
print(f"Normalized Mutual Information (NMI): {hier_comparison['normalized_mutual_info']}")
print(f"K-Means Silhouette: {hier_comparison['kmeans']['silhouette_score']} | DB: {hier_comparison['kmeans']['davies_bouldin_index']}")
print(f"Hierarchical Silhouette: {hier_comparison['hierarchical']['silhouette_score']} | DB: {hier_comparison['hierarchical']['davies_bouldin_index']}")
print(f"
Conclusion: {hier_comparison['comparison_summary']}")


### 2.4 ANOVA & Kruskal-Wallis Feature Separation Power Across Clusters
To identify which biometeorological variables drive cluster boundary separation most strongly, we conduct both parametric One-Way Analysis of Variance (ANOVA $F$-statistic) and non-parametric Kruskal-Wallis ($H$-statistic) across all clusters.

> **Methodological Note:** These statistical tests evaluate *feature-level discriminative separation across cluster boundaries*, **not model predictive accuracy** (since clustering is unsupervised).


In [ ]:
separation_stats = engineer.calculate_feature_cluster_separation(
    engineered_df, cluster_col='cluster', feature_cols=feature_cols
)
sep_df = pd.DataFrame(separation_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ANOVA F-statistic
sns.barplot(data=sep_df.sort_values('f_statistic', ascending=False), x='f_statistic', y='feature', ax=axes[0], palette='Blues_r')
axes[0].set_title("Parametric ANOVA F-Statistic", fontsize=12, fontweight='bold')
axes[0].set_xlabel("F-Statistic")

# Kruskal-Wallis H-statistic
sns.barplot(data=sep_df.sort_values('kruskal_statistic', ascending=False), x='kruskal_statistic', y='feature', ax=axes[1], palette='Oranges_r')
axes[1].set_title("Non-Parametric Kruskal-Wallis H-Statistic", fontsize=12, fontweight='bold')
axes[1].set_xlabel("H-Statistic")

plt.tight_layout()
plt.show()

print("Feature discriminative separation ranking:")
sep_df[['feature', 'f_statistic', 'kruskal_statistic', 'p_value', 'separation_score']]


## Part 3: PCA Visualisation, Heat-Stress Profiles & Priority Mapping (CO5)
In this section, we apply **Principal Component Analysis (PCA)** to project the 8-dimensional feature space into 2 orthogonal principal components, interpret the feature loading vectors (biplot), construct **Heat-Stress Vulnerability Profiles** (Low, Moderate, High, Extreme), and visualize the geographical priority map.


In [ ]:
pca_reducer = PCAReducer(n_components=2)
pca_output = pca_reducer.fit_transform(X_scaled, feature_cols)

coords_2d = np.array(pca_output['coordinates_2d'])
exp_var = pca_output['explained_variance_ratio']

print(f"PC1 Variance Explained: {exp_var[0]*100:.2f}%")
print(f"PC2 Variance Explained: {exp_var[1]*100:.2f}%")
print(f"Total 2D Variance Captured: {pca_output['total_variance_explained_2d']}%")

# Create PCA DataFrame
pca_df = pd.DataFrame({
    'PC1': coords_2d[:, 0],
    'PC2': coords_2d[:, 1],
    'cluster': labels_opt,
    'station': engineered_df['NAME'],
    'heat_index': engineered_df['heat_index_c']
})


### 3.1 PCA 2D Cluster Visualization & Biplot Loadings
We plot the 2D cluster projections alongside the biplot arrows representing the direction and magnitude of the meteorological features.


In [ ]:
plt.figure(figsize=(12, 8))
palette = ['#10B981', '#F59E0B', '#EF4444', '#7C3AED', '#3B82F6']
sns.scatterplot(
    data=pca_df, x='PC1', y='PC2', hue='cluster',
    palette=palette[:best_k], s=65, alpha=0.85, edgecolor='none'
)

# Plot feature loading biplot vectors
loadings = pca_output['feature_loadings']
scale_factor = 3.5  # visual scaling for arrow visibility
for load in loadings:
    plt.arrow(0, 0, load['pc1_loading']*scale_factor, load['pc2_loading']*scale_factor,
              color='black', alpha=0.6, width=0.03, head_width=0.12)
    plt.text(load['pc1_loading']*scale_factor*1.15, load['pc2_loading']*scale_factor*1.15,
             load['feature'], color='#1E293B', fontsize=10, fontweight='bold')

plt.title(f"PCA 2D Biplot Cluster Projection (Variance Explained: {pca_output['total_variance_explained_2d']}%)", fontsize=14, fontweight='bold')
plt.xlabel(f"Principal Component 1 ({exp_var[0]*100:.1f}% Variance)")
plt.ylabel(f"Principal Component 2 ({exp_var[1]*100:.1f}% Variance)")
plt.legend(title="Cluster ID", loc='best')
plt.axhline(0, color='gray', linestyle='--', alpha=0.3)
plt.axvline(0, color='gray', linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()


### 3.2 Interpreting Heat-Stress Vulnerability Profiles
We aggregate the physical unscaled biometeorological variables for each cluster, calculate the thermal severity index, and map each cluster to human-interpretable vulnerability tiers and civil defense recommendations.


In [ ]:
profiles = VulnerabilityProfiler.generate_profiles(engineered_df, cluster_col='cluster')

profile_summary = []
for p in profiles:
    profile_summary.append({
        'Cluster': p['cluster_id'],
        'Title': p['title'],
        'Tier': p['vulnerability_tier'],
        'Share (%)': p['percentage'],
        'Mean Temp (°C)': p['avg_temp_c'],
        'Max Temp (°C)': p['max_temp_c'],
        'Heat Index (°C)': p['heat_index_c'],
        'Dew Point (°C)': p['dew_point_c'],
        'RH (%)': p['relative_humidity_pct'],
        'Wind (km/h)': p['wind_speed_kmh']
    })

pd.DataFrame(profile_summary)


### 3.3 Public Health Mitigation Protocols per Profile
For each identified profile, we display the mandated disaster risk reduction protocol.


In [ ]:
for p in profiles:
    print(f"\n=======================================================")
    print(f"📌 {p['title'].upper()} ({p['percentage']}% of Stations)")
    print(f"Priority Level: {p['priority_level']}")
    print(f"Thermodynamic Profile: {p['description']}")
    print("Actionable Mitigation Protocols:")
    for act in p['actionable_recommendations']:
        print(f"  • {act}")


### 3.4 Regional Priority Mapping (Geographical Coordinates)
Finally, we display the geographical distribution of weather stations color-coded by vulnerability profile.


In [ ]:
# Plot regional priority map using station latitude and longitude
plt.figure(figsize=(12, 10))

# Create color mapping from profiles
tier_colors = {'Low': '#10B981', 'Moderate': '#F59E0B', 'High': '#EF4444', 'Extreme': '#7C3AED'}
cluster_to_tier = {p['cluster_id']: p['vulnerability_tier'] for p in profiles}
engineered_df['tier'] = engineered_df['cluster'].map(cluster_to_tier)

# Aggregate station level coordinates
station_geo = engineered_df.groupby('STATION').agg({
    'NAME': 'first',
    'LATITUDE': 'first',
    'LONGITUDE': 'first',
    'heat_index_c': 'max',
    'tier': lambda x: x.mode()[0]
}).reset_index()

for tier in ['Low', 'Moderate', 'High', 'Extreme']:
    subset = station_geo[station_geo['tier'] == tier]
    if len(subset) > 0:
        plt.scatter(
            subset['LONGITUDE'], subset['LATITUDE'],
            c=tier_colors[tier], label=f"{tier} Priority ({len(subset)} stations)",
            s=120, edgecolors='black', linewidth=0.7, alpha=0.9
        )

plt.title("Regional Heat-Stress Vulnerability Priority Map (NOAA GSOD)", fontsize=14, fontweight='bold')
plt.xlabel("Longitude (°E)")
plt.ylabel("Latitude (°N)")
plt.legend(loc='lower right', frameon=True, fontsize=11)
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


### 3.5 Continuous Heat Stress Index (HSI: 0–100) Distribution
Unlike discrete cluster assignments, the continuous HSI score (0 to 100) allows fine-grained prioritisation within each cluster tier.


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=engineered_df, x='heat_stress_index', hue='cluster', palette=palette[:best_k], kde=True, bins=25, alpha=0.6)
plt.title("Distribution of Continuous Heat Stress Index (HSI: 0-100) Across Clusters", fontsize=13, fontweight='bold')
plt.xlabel("Continuous Heat Stress Index (HSI)")
plt.ylabel("Observation Frequency")
plt.axvline(50, color='gray', linestyle=':', label='Moderate Baseline (50)')
plt.axvline(75, color='red', linestyle='--', label='Severe Threshold (75)')
plt.legend()
plt.tight_layout()
plt.show()


### 3.6 Multi-Year Temporal Cluster Shifts (2022–2025)
Heat-stress profiles fluctuate across consecutive summer seasons. Here we inspect the progression of cluster shares across the 4-year period.


In [ ]:
annual_shifts = TemporalAnalyzer.analyze_annual_cluster_shifts(engineered_df, cluster_col='cluster')
annual_df = pd.DataFrame([{
    'Year': s['year'],
    'Total Obs': s['total_observations'],
    'Extreme Share (%)': s['high_risk_share_pct'],
    'Avg Temp (°C)': s['mean_temp_c'],
    'Avg Heat Index (°C)': s['mean_heat_index_c']
} for s in annual_shifts])

print("=== MULTI-YEAR ANNUAL CLUSTER TRANSITION TRENDS ===")
print(annual_df)


### 3.7 Station Diagnostic Explainability Drilldown
For any monitored station, we can inspect contributing biometeorological percentiles to answer **"Why is this region high priority?"**


In [ ]:
# Explain a high-priority station
sample_station_id = engineered_df[engineered_df['cluster'] == profiles[-1]['cluster_id']]['STATION'].iloc[0]
station_diag = StationExplainer.explain_station(sample_station_id, engineered_df, profiles)

print(f"=== STATION DIAGNOSTIC REPORT: {station_diag['name']} ({station_diag['station_id']}) ===")
print(f"Priority Level: {station_diag['priority_level']}")
print(f"Heat Stress Index (HSI): {station_diag['heat_stress_index']} / 100")
print(f"Diagnostic Rationale:\n{station_diag['explanation']}\n")
print("Contributing Indicators:")
for ind in station_diag['key_contributing_indicators']:
    print(f" - {ind['name']}: {ind['value']} (Percentile: {ind['percentile']}%, Importance: {ind['importance']}%)")


### 3.8 Automated AI Insights & Decision Intelligence Synthesis
We run the Natural Language Generation (NLG) engine to synthesize the multi-dimensional clustering results into an executive policy brief.


In [ ]:
summary_dict = {
    'total_observations': len(engineered_df),
    'unique_stations': int(engineered_df['STATION'].nunique()),
    'features_used': feature_cols,
    'optimal_k': best_k,
    'best_silhouette_score': optimal_result['best_evaluation']['silhouette_score'],
    'mean_heat_index_c': round(float(engineered_df['heat_index_c'].mean()), 1),
    'max_recorded_temp_c': round(float(engineered_df['max_temp_c'].max()), 1),
    'min_recorded_temp_c': round(float(engineered_df['min_temp_c'].min()), 1)
}

ai_brief = AIInsightsEngine.generate_executive_insights(
    summary=summary_dict,
    profiles=profiles,
    optimal_k_data=optimal_result,
    feature_separation=separation_stats,
    temporal_summary=annual_shifts
)

print(f"=== EXECUTIVE BRIEF: {ai_brief['title']} ===")
print(f"Model Confidence: {ai_brief['model_confidence']}\n")
print("Key Executive Takeaways:")
for take in ai_brief['key_takeaways']:
    print(f" • {take}")
print(f"
Temporal Trend Narrative:
{ai_brief['temporal_trend_narrative']}")
print(f"
Civil Defense Brief:
{ai_brief['civil_defense_brief']}")


## Conclusion & Academic Course Outcomes Verification

### Course Outcomes (CO) Attainment:
1. **CO2 (Data Preprocessing, Cleaning & Scaling)**:
   - Ingested raw NOAA GSOD observations across 4 summer seasons (2022–2025), detected and imputed NOAA missing numeric flags (`9999.9`) using robust median imputation, converted imperial measures to SI metric units ($^\circ\text{C}$, $\text{km/h}$, $\text{hPa}$), and standardized features with `StandardScaler` to remove scale bias.
   - Formulated scientifically grounded biometeorological variables including **Magnus-Tetens Relative Humidity ($RH$)**, **Rothfusz NOAA Heat Index ($HI$)**, **Diurnal Temperature Range ($DTR$)**, **Canadian Humidex**, and continuous **Heat Stress Index (HSI: 0–100)**.

2. **CO3 (Partition Clustering, Model Validation & Hierarchical Comparison)**:
   - Executed systematic K-Means partition clustering across $K \in [2, 8]$.
   - Evaluated models using Within-Cluster Sum of Squares (**WCSS Elbow Method**), **Silhouette Coefficients**, **Davies–Bouldin Indices**, and **Calinski–Harabasz Indices**.
   - Built an automated multi-criteria decision engine selecting the mathematically optimal cluster configuration ($K=4$) with explicit quantitative justification.
   - Satisfied the **Hierarchical Comparison** syllabus requirement by running Agglomerative Hierarchical Clustering (Ward's linkage), proving substantial cluster stability via Cophenetic correlation ($r > 0.70$) and Adjusted Rand Index.
   - Quantified discriminative power using One-Way ANOVA $F$-statistics across all input features.

3. **CO5 (Dimensionality Reduction, Interpretation & Priority Mapping)**:
   - Applied **Principal Component Analysis (PCA)** to compute 2D projections explaining the vast majority of dataset variance, accompanied by a feature loading biplot exposing the orthogonal thermodynamic axes (Thermal Intensity vs Atmospheric Moisture).
   - Translated mathematical cluster centroids into actionable **Heat-Stress Vulnerability Profiles** (Profiles A through D) linked with targeted public health mitigation protocols.
   - Implemented multi-year temporal tracking (2022–2025) and individual station diagnostic explainability.
   - Successfully rendered a regional **Priority Map** geographically highlighting acute heat-stress hotspots to inform emergency response and cooling infrastructure allocation.
